In [12]:
# Li et al. Market Making Strategy Backtest
# This notebook demonstrates backtesting of the Li et al. market making strategy
# with order book pressure and news signal integration

# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)


## Imports

In [13]:
from hummingbot.strategy_v2.utils.distributions import Distributions
from controllers.market_making.pmm_Li import PMMLIControllerConfig
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from core.backtesting import BacktestingEngine
import datetime
from decimal import Decimal

In [14]:

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=False)

## BOT Parameter Config

In [15]:

# Controller configuration
connector_name = "binance"
trading_pair = "POL-USDT"
total_amount_quote = 1000 # Total amount in quote currency (USDT)
take_profit = 0.02
stop_loss = 0.01
trailing_stop_activation_price = 0.015
trailing_stop_trailing_delta = 0.07
time_limit = 60 * 60 * 2
executor_refresh_time = 60 * 6
cooldown_time = 600

# PMM Li configuration
volatility_window = 60 * 60 * 24  # 24 hours

# order book levels
sell_order_book_levels = 4
buy_order_book_levels = 4

# Backtesting configuration
start = int(datetime.datetime(2025, 5, 25).timestamp())
end = int(datetime.datetime(2025, 5, 26).timestamp())
backtesting_resolution = "1m"

## Bot Risk Management

In [16]:
config = PMMLIControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    candles_connector=connector_name,
    candles_trading_pair=trading_pair, 
    volatility_window=volatility_window,
    total_amount_quote=Decimal(total_amount_quote),
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(
        activation_price=Decimal(trailing_stop_activation_price),
        trailing_delta=Decimal(trailing_stop_trailing_delta)
    ),
    time_limit=time_limit,
    executor_refresh_time=executor_refresh_time,
    cooldown_time=cooldown_time,
    buy_spreads=[0.5, 1.0, 1.5, 2.0],
    sell_spreads=[0.5, 1.0, 1.5, 2.0],
    buy_amounts_pct=[Decimal(0.1), Decimal(0.2), Decimal(0.3), Decimal(0.4)],
    sell_amounts_pct=[Decimal(0.1), Decimal(0.2), Decimal(0.3), Decimal(0.4)],
)

In [17]:
# ===== BACKTESTING EXECUTION =====

# Run backtesting
backtesting_result = await backtesting.run_backtesting(config, start, end, backtesting_resolution)

print("Backtesting completed!")


Processing 481 and Index(['timestamp', 'open', 'high', 'low', 'close', 'volume',
       'quote_asset_volume', 'n_trades', 'taker_buy_base_volume',
       'taker_buy_quote_volume'],
      dtype='object')
Backtesting completed!


In [18]:
backtesting_result.data

{'executors': [ExecutorInfo(id='CA3chabMyt1FANkycGkyzAQR17B7RVAw2mB8Z6rdd3R5', timestamp=1748126340.0, type='position_executor', status=<RunnableStatus.TERMINATED: 4>, config=PositionExecutorConfig(id='CA3chabMyt1FANkycGkyzAQR17B7RVAw2mB8Z6rdd3R5', type='position_executor', timestamp=1748126340.0, controller_id='main', trading_pair='POL-USDT', connector_name='binance', side=<TradeType.BUY: 1>, entry_price=Decimal('0.2322227864034675495133773421'), amount=Decimal('215.3104816903247769822145841'), triple_barrier_config=TripleBarrierConfig(stop_loss=Decimal('0.01000000000000000020816681711721685132943093776702880859375'), take_profit=Decimal('0.0200000000000000004163336342344337026588618755340576171875'), time_limit=7200, trailing_stop=TrailingStop(activation_price=Decimal('0.01499999999999999944488848768742172978818416595458984375'), trailing_delta=Decimal('0.070000000000000006661338147750939242541790008544921875')), open_order_type=<OrderType.LIMIT: 2>, take_profit_order_type=<OrderType

In [19]:

# ===== RESULTS ANALYSIS =====

print("\n=== BACKTESTING RESULTS SUMMARY ===")
print(backtesting_result.get_results_summary())



=== BACKTESTING RESULTS SUMMARY ===

Net PNL: $3.32 (0.33%) | Max Drawdown: $-8.52 (-0.85%)
Total Volume ($): 15502.23 | Sharpe Ratio: 0.21 | Profit Factor: 1.16
Total Executors: 491 | Accuracy Long: 0.43 | Accuracy Short: 0.63
Close Types: Take Profit: 0 | Stop Loss: 12 | Time Limit: 63 |
             Trailing Stop: 0 | Early Stop: 416



In [20]:

# Display the backtesting figure
print("\n=== PERFORMANCE VISUALIZATION ===")
backtesting_result.get_backtesting_figure()



=== PERFORMANCE VISUALIZATION ===


In [21]:

# ===== TESTING DIFFERENT NEWS SIGNALS ===

print("\n=== TESTING DIFFERENT NEWS SIGNALS ===")
print("You can modify the news_signal_port parameter to test different scenarios:")
print("- Positive values (0 to 1): Bullish news signal")
print("- Negative values (-1 to 0): Bearish news signal") 
print("- Zero (0): Neutral signal")

# Example configurations for different news scenarios
news_scenarios = [
    ("Neutral", 0.0),
    ("Bullish", 0.5), 
    ("Bearish", -0.5),
    ("Strong Bullish", 0.8),
    ("Strong Bearish", -0.8)
]

print("\n=== NEWS SIGNAL SCENARIOS ===")
for scenario_name, signal_value in news_scenarios:
    print(f"{scenario_name}: news_signal_port = {signal_value}")

print("\n=== OPTIMIZATION SUGGESTIONS ===")
print("1. Tune μ (mu_scaling_factor) based on order book depth")
print("2. Adjust η (eta_scaling_factor) based on news signal reliability")
print("3. Optimize obp_threshold for signal activation")
print("4. Test different volatility_window sizes")
print("5. Experiment with obp_levels and obp_history_snapshots")

print("\n=== NEXT STEPS ===")
print("1. Run parameter sweeps to optimize μ and η values")
print("2. Implement real-time news signal feed")
print("3. Add more sophisticated volatility models")
print("4. Test on different market conditions and trading pairs")
print("5. Compare performance against standard PMM strategy")


=== TESTING DIFFERENT NEWS SIGNALS ===
You can modify the news_signal_port parameter to test different scenarios:
- Positive values (0 to 1): Bullish news signal
- Negative values (-1 to 0): Bearish news signal
- Zero (0): Neutral signal

=== NEWS SIGNAL SCENARIOS ===
Neutral: news_signal_port = 0.0
Bullish: news_signal_port = 0.5
Bearish: news_signal_port = -0.5
Strong Bullish: news_signal_port = 0.8
Strong Bearish: news_signal_port = -0.8

=== OPTIMIZATION SUGGESTIONS ===
1. Tune μ (mu_scaling_factor) based on order book depth
2. Adjust η (eta_scaling_factor) based on news signal reliability
3. Optimize obp_threshold for signal activation
4. Test different volatility_window sizes
5. Experiment with obp_levels and obp_history_snapshots

=== NEXT STEPS ===
1. Run parameter sweeps to optimize μ and η values
2. Implement real-time news signal feed
3. Add more sophisticated volatility models
4. Test on different market conditions and trading pairs
5. Compare performance against standard 